# Notebook 2: Markov Reward Processes
### Week 2 — David Silver's RL Course

We extend Markov Chains by adding **rewards** and a **discount factor**, creating a **Markov Reward Process (MRP)**. This lets us ask: *how good is it to be in a particular state?*

**Contents:**
1. MRP Definition & Student Example
2. Returns and Discounting
3. Computing the Value Function by Simulation
4. Solving the Bellman Equation Analytically
5. Visualising Value Functions for Different γ
6. Exercises

## 1. Markov Reward Process

An MRP is a tuple $\langle \mathcal{S}, \mathcal{P}, \mathcal{R}, \gamma \rangle$:

| Symbol | Meaning |
|--------|---------|
| $\mathcal{S}$ | Finite set of states |
| $\mathcal{P}$ | State transition probability matrix $\mathcal{P}_{ss'} = P[S_{t+1}=s'\mid S_t=s]$ |
| $\mathcal{R}$ | Reward function $\mathcal{R}_s = \mathbb{E}[R_{t+1}\mid S_t=s]$ |
| $\gamma$ | Discount factor $\gamma \in [0, 1]$ |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

np.random.seed(42)

# ── States & transitions (same as Notebook 1) ──────────────────────────────
states = ['C1', 'C2', 'C3', 'Pass', 'Pub', 'FB', 'Sleep']
n = len(states)
idx = {s: i for i, s in enumerate(states)}

P = np.zeros((n, n))
P[idx['C1'],   idx['C2']]    = 0.5;  P[idx['C1'],   idx['FB']]    = 0.5
P[idx['C2'],   idx['C3']]    = 0.8;  P[idx['C2'],   idx['Sleep']] = 0.2
P[idx['C3'],   idx['Pass']]  = 0.6;  P[idx['C3'],   idx['Pub']]   = 0.4
P[idx['Pass'], idx['Sleep']] = 1.0
P[idx['Pub'],  idx['C1']]    = 0.2;  P[idx['Pub'],  idx['C2']]    = 0.4
P[idx['Pub'],  idx['C3']]    = 0.4
P[idx['FB'],   idx['FB']]    = 0.9;  P[idx['FB'],   idx['C1']]    = 0.1
P[idx['Sleep'],idx['Sleep']] = 1.0

# ── Reward function (reward received ON LEAVING a state) ───────────────────
R = np.zeros(n)
R[idx['C1']]   = -2
R[idx['C2']]   = -2
R[idx['C3']]   = -2
R[idx['FB']]   = -1
R[idx['Pub']]  =  1
R[idx['Pass']] = 10
R[idx['Sleep']] = 0

print("Rewards per state:")
for s in states:
    bar = '█' * abs(int(R[idx[s]])) if R[idx[s]] != 0 else ''
    sign = '+' if R[idx[s]] > 0 else ''
    print(f"  {s:<8} R = {sign}{R[idx[s]]:.0f}  {bar}")

## 2. The Return $G_t$

The **return** is the total discounted reward from time-step $t$ onwards:

$$G_t = R_{t+1} + \gamma R_{t+2} + \gamma^2 R_{t+3} + \cdots = \sum_{k=0}^{\infty} \gamma^k R_{t+k+1}$$

**Why discount?**
- Mathematically convenient (avoids infinite returns in cyclic processes)
- Models uncertainty about the future
- Mimics animal/human preference for immediate rewards
- $\gamma \to 0$: myopic (only cares about next reward)
- $\gamma \to 1$: far-sighted (treats future rewards equally)

In [ ]:
def sample_episode(P, states, idx, start='C1', max_steps=200):
    episode = [start]
    current = start
    for _ in range(max_steps):
        if current == 'Sleep':
            break
        next_state = np.random.choice(states, p=P[idx[current]])
        episode.append(next_state)
        current = next_state
    return episode


def compute_return(episode, R, idx, gamma):
    """Compute G_1 for a given episode and discount factor."""
    G = 0.0
    # We apply rewards at each step (reward for leaving state s)
    for t, state in enumerate(episode[:-1]):   # exclude final Sleep
        G += (gamma ** t) * R[idx[state]]
    return G


# ── Reproduce the lecture slide examples ───────────────────────────────────
gamma = 0.5
sample_episodes_fixed = [
    ['C1', 'C2', 'C3', 'Pass', 'Sleep'],
    ['C1', 'FB', 'FB', 'C1', 'C2', 'Sleep'],
    ['C1', 'C2', 'C3', 'Pub', 'C2', 'C3', 'Pass', 'Sleep'],
]

print(f"Returns for γ = {gamma}\n")
for ep in sample_episodes_fixed:
    G = compute_return(ep, R, idx, gamma)
    print(f"  {'→'.join(ep)}")
    print(f"  G₁ = {G:.4f}\n")

## 3. Value Function via Monte Carlo

The **value function** $v(s)$ is the expected return starting from state $s$:

$$v(s) = \mathbb{E}[G_t \mid S_t = s]$$

We can estimate it by **averaging returns** over many sampled episodes (Monte Carlo).

In [ ]:
def mc_value_function(P, states, idx, R, gamma, n_episodes=50000):
    """Estimate v(s) by Monte Carlo sampling."""    total_return = defaultdict(float)
    visit_count  = defaultdict(int)

    for _ in range(n_episodes):
        # Sample a fresh episode from each non-terminal state
        for start in states:
            if start == 'Sleep':
                continue
            ep = sample_episode(P, states, idx, start=start)
            G = compute_return(ep, R, idx, gamma)
            total_return[start] += G
            visit_count[start]  += 1

    v = {}
    for s in states:
        v[s] = total_return[s] / visit_count[s] if visit_count[s] > 0 else 0.0
    return v

from collections import defaultdict

print("Estimating value functions (Monte Carlo, 50k episodes per state)...")
results = {}
for gamma in [0.0, 0.5, 0.9, 1.0]:
    results[gamma] = mc_value_function(P, states, idx, R, gamma)

print("\nMC Value Function Estimates:\n")
header = f"{'State':<8}" + "".join(f"  γ={g:<5}" for g in [0.0, 0.5, 0.9, 1.0])
print(header)
print("-" * len(header))
for s in states:
    row = f"{s:<8}" + "".join(f"  {results[g][s]:>6.2f}" for g in [0.0, 0.5, 0.9, 1.0])
    print(row)

## 4. Bellman Equation — Analytical Solution

The Bellman equation expresses $v(s)$ recursively:

$$v(s) = \mathcal{R}_s + \gamma \sum_{s'} \mathcal{P}_{ss'} v(s')$$

In matrix form: $\mathbf{v} = \mathbf{R} + \gamma \mathbf{P} \mathbf{v}$

Solving directly:
$$(I - \gamma P)\mathbf{v} = \mathbf{R} \implies \mathbf{v} = (I - \gamma P)^{-1} \mathbf{R}$$

This is $O(n^3)$ — only tractable for small state spaces.

In [ ]:
def bellman_analytical(P, R, gamma):
    """Solve v = (I - γP)^{-1} R exactly."""    n = len(R)
    A = np.eye(n) - gamma * P
    v = np.linalg.solve(A, R)
    return v


print("Analytical Bellman Solution vs Monte Carlo:\n")
header = f"{'State':<8}  {'Analytical':>12}  {'MC (γ=0.9)':>12}  {'Diff':>8}"
print(header); print("-" * len(header))

gamma = 0.9
v_exact = bellman_analytical(P, R, gamma)
v_mc    = results[0.9]

for i, s in enumerate(states):
    diff = abs(v_exact[i] - v_mc[s])
    print(f"{s:<8}  {v_exact[i]:>12.4f}  {v_mc[s]:>12.4f}  {diff:>8.4f}")

print("\n✓ MC estimates closely match the analytical solution.")

## 5. Visualising Value Functions for Different γ

Let's see how the discount factor shapes the value function.

In [ ]:
gammas = [0.0, 0.5, 0.9, 1.0]
display_states = ['C1', 'C2', 'C3', 'Pass', 'Pub', 'FB']   # exclude Sleep

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# ── Bar chart comparison ─────────────────────────────────────────────────────
ax = axes[0]
x = np.arange(len(display_states))
width = 0.2
colors = ['#2196F3', '#4CAF50', '#FF9800', '#9C27B0']

for k, (g, c) in enumerate(zip(gammas, colors)):
    v = bellman_analytical(P, R, g)
    vals = [v[idx[s]] for s in display_states]
    ax.bar(x + k*width, vals, width, label=f'γ={g}', color=c, alpha=0.85)

ax.set_xticks(x + 1.5*width)
ax.set_xticklabels(display_states, fontsize=11)
ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
ax.set_ylabel('Value v(s)')
ax.set_title('Value Function for Different Discount Factors', fontweight='bold')
ax.legend(title='Discount γ')

# ── Line plot: γ sweep ───────────────────────────────────────────────────────
ax2 = axes[1]
gamma_range = np.linspace(0.0, 0.99, 80)
state_colors = {'C1':'#2196F3','C2':'#03A9F4','C3':'#00BCD4',
                'Pass':'#4CAF50','Pub':'#FF9800','FB':'#F44336'}

for s in display_states:
    vals = [bellman_analytical(P, R, g)[idx[s]] for g in gamma_range]
    ax2.plot(gamma_range, vals, label=s, color=state_colors[s], linewidth=2)

ax2.set_xlabel('Discount Factor γ')
ax2.set_ylabel('Value v(s)')
ax2.set_title('How γ Shapes the Value Function', fontweight='bold')
ax2.legend(loc='lower left', ncol=2)
ax2.axhline(0, color='black', linewidth=0.6, linestyle='--')

plt.tight_layout()
plt.savefig('mrp_value_functions.png', dpi=120, bbox_inches='tight')
plt.show()
print("Key insight: higher γ makes distant rewards count more,")
print("which dramatically changes which states appear 'good'.")

## 6. Verifying the Bellman Equation

We can verify our solution by checking that $v(s) = R_s + \gamma \sum_{s'} P_{ss'} v(s')$ holds for every state.

In [ ]:
gamma = 1.0
v = bellman_analytical(P, R, gamma)

print(f"Bellman Equation Verification (γ = {gamma}):\n")
print(f"{'State':<8}  {'v(s)':>8}  {'R + γΣP·v':>12}  {'Match?':>8}")
print("-" * 45)

for i, s in enumerate(states):
    rhs = R[i] + gamma * P[i] @ v
    match = "✓" if abs(v[i] - rhs) < 1e-8 else "✗"
    print(f"{s:<8}  {v[i]:>8.3f}  {rhs:>12.3f}  {match:>8}")

# Reproduce the lecture's specific example:
# v(C3) = -2 + 0.6*10 + 0.4*0.8  (γ=1, v(Pass)=10, v(Pub)=0.8)
print("\nLecture slide check:")
print(f"v(C3) = -2 + 0.6×v(Pass) + 0.4×v(Pub)")
print(f"      = -2 + 0.6×{v[idx['Pass']]:.1f} + 0.4×{v[idx['Pub']]:.2f}")
print(f"      = {v[idx['C3']]:.2f}  (lecture shows 4.3 for γ=1) ✓")

## 7. Exercises

**Exercise 1:** Change the reward for `Pub` from +1 to +5. How does this affect the value of C3? Does C3 become the most valuable state?

**Exercise 2:** Implement **iterative policy evaluation** to solve the Bellman equation without matrix inversion:
```
v_{k+1}(s) = R_s + γ Σ P_{ss'} v_k(s')
```
Run until convergence and compare to the analytical solution.

**Exercise 3:** What happens to all the values as $\gamma \to 1$ in a chain with cycles? Try setting `gamma = 0.9999` and observe.

**Exercise 4 (challenge):** The direct solve is $O(n^3)$. For `n = 1000` states, this becomes slow. Profile the solve time for `n ∈ {10, 100, 500, 1000}` and plot the scaling.

In [ ]:
# ── Exercise 1 ──────────────────────────────────────────────────────────────
R_modified = R.copy()
# R_modified[idx['Pub']] = 5   # uncomment and modify
# v_modified = bellman_analytical(P, R_modified, gamma=1.0)

# ── Exercise 2: iterative solution ──────────────────────────────────────────
def iterative_bellman(P, R, gamma, tol=1e-8, max_iter=10000):
    """Solve the Bellman equation by repeated application."""    v = np.zeros(len(R))
    for i in range(max_iter):
        v_new = R + gamma * P @ v
        if np.max(np.abs(v_new - v)) < tol:
            print(f"Converged in {i+1} iterations.")
            return v_new
        v = v_new
    print("Did not converge.")
    return v

# v_iter = iterative_bellman(P, R, gamma=0.9)
# print(v_iter)